<a href="https://colab.research.google.com/github/sumitp2703/FMML/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Implementation  Of RNN**

**Recurrent Neural Networks (RNNs)** are a type of neural network designed to handle sequential data by maintaining a state or memory of previous inputs. They are widely used in tasks where the order and context of data are crucial, such as natural language processing, speech recognition, and time series prediction.

**Structure of RNNs**
Recurrent Connections: RNNs have loops that allow information to persist. This loop gives them the ability to use information about previous elements in a sequence to inform the prediction of the next element.

**Types of RNNs:**

*   **Vanilla RNN:** Basic form where the hidden state is a function of the current input and the previous hidden state.
*   **Long Short-Term Memory (LSTM):** Designed to overcome the vanishing gradient problem of vanilla RNNs by introducing specialized memory cells and gating mechanisms.
* **Gated Recurrent Unit (GRU):** A simpler variant of LSTM that also uses gating mechanisms to control the flow of information.  










In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


In [ ]:
# Generate sequential data
data = np.array([[[1.], [2.], [3.], [4.], [5.]]])  # Shape: (1, 5, 1) - 1 sequence, 5 time steps, 1 feature
data = torch.tensor(data, dtype=torch.float32)


In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(1, x.size(0), self.hidden_size)

        # Forward propagate RNN
        out, _ = self.rnn(x, h0)

        # Decode the hidden state of the last time step
        out = self.fc(out[:, -1, :])
        return out


In [ ]:
input_size = 1
hidden_size = 32
output_size = 1

model = SimpleRNN(input_size, hidden_size, output_size)


In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [ ]:
num_epochs = 1000

for epoch in range(num_epochs):
    # Forward pass
    outputs = model(data)
    loss = criterion(outputs, data.view(-1, 1))

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


/usr/local/lib/python3.10/dist-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([5, 1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [100/1000], Loss: 2.0000
Epoch [200/1000], Loss: 2.0000
Epoch [300/1000], Loss: 2.0000
Epoch [400/1000], Loss: 2.0000
Epoch [500/1000], Loss: 2.0000
Epoch [600/1000], Loss: 2.0000
Epoch [700/1000], Loss: 2.0000
Epoch [800/1000], Loss: 2.0000
Epoch [900/1000], Loss: 2.0000
Epoch [1000/1000], Loss: 2.0000


In [ ]:
# Test the model
with torch.no_grad():
    test_data = torch.tensor([[[6.], [7.], [8.], [9.], [10.]]], dtype=torch.float32)
    predicted = model(test_data)
    print(f'Predicted next sequence: {predicted.squeeze().numpy()}')


Predicted next sequence: 3.036635160446167


In [ ]:
import torch
import torch.nn as nn

class RNNModel(nn.Module):
    def __init__(self, image_feature_size, graph_feature_size, hidden_size, num_classes):
        super(RNNModel, self).__init__()
        self.image_feature_size = image_feature_size
        self.graph_feature_size = graph_feature_size
        self.hidden_size = hidden_size

        # Define layers for image processing
        self.image_fc = nn.Linear(image_feature_size, hidden_size)

        # Define layers for graph processing
        self.graph_fc = nn.Linear(graph_feature_size, hidden_size)

        # RNN layer
        self.rnn = nn.RNN(input_size=hidden_size, hidden_size=hidden_size, batch_first=True)

        # Output layer
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, image_input, graph_input):
        # Image processing
        image_output = torch.relu(self.image_fc(image_input))

        # Graph processing
        graph_output = torch.relu(self.graph_fc(graph_input))

        # Combine image and graph features
        combined_features = image_output + graph_output

        # RNN input should have shape (batch_size, sequence_length, input_size)
        rnn_input = combined_features.unsqueeze(1)

        # RNN layer
        rnn_output, _ = self.rnn(rnn_input)

        # Final classification output
        output = self.fc(rnn_output[:, -1, :])  # Take the last time step's output
        return output
